In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-8: Who supplies the weights?

## RFT Alternative methods, part 2 of the RLLM series


## Where this session fits

> **RLLM-7 said where samples come from. Every method turns samples into the same kind of update; only the weights differ. This session: axes 2 and 3, where the weights come from and how fine they are.**

- **Part 2: RFT Alternative methods**: alternatives to the PPO and GRPO you know, each placed by three axes
- **Axes**: 1 whose samples (RLLM-7), **2 who supplies the weights: a reference or a reward (here)**, **3 how fine (here)**
- **Weight**: the multiplier an update puts on a token
- **Reference**: what the update imitates (a dataset's token or a teacher's distribution), not RLLM-4's *reference model*
- **Position**: session 8 of 14, after RLLM-7 (Whose samples? Off-policy vs. on-policy)
- **Goal of Part 2**: a **map** that places any method by the three axes

<img src="images/series_map_8.png" width=55%>


## This session: RLLM-8

- **Question**: where do an update's **weights** come from (a **reference** or a **reward**), and how **fine** are they?
- **By the end you can**:
  - say that the methods share **one gradient form** and differ in how the **weights** are chosen
  - say what a **push** is (the direction an update moves the model's probabilities), and why own completions with **equal weights** carry none on average
  - say for a method whether its weights come from a **reference** or a **reward**
  - say how fine a method's weights are, and why an **episode reward** makes credit assignment hard


## Map: the grid this session fills

> **RLLM-7 gave the rows (whose samples). This session adds the columns (who supplies the weights) and, inside each box, how fine the weights are.**

<img src="images/rllm8_map.png" alt="an empty grid: rows whose samples, columns a reference or a reward supplies the weights, with the sections that cover each part" width=50%>

- This is the **plan**: the cells are filled in the **Placements** section
- **Gradient form** and **Push** define two words
  - a **weight**: the multiplier on a token
  - a **push**: the direction it moves that token's probability
- **No-weights case** shows why the weights must come from outside
- **Not RLLM-7's grid**: that crossed whose samples with which prefix. This one crosses whose samples with who supplies the weights
- **Weight source** gives the columns; **Weight granularity** the second line in each box


## Gradient form: all methods share one form

> **Why a common form? It is one yardstick: every method pushes some tokens' probabilities up or down, and the weight says how hard. Comparing methods then means comparing weights.**
>
> **All methods share this sum: a weight times the gradient of $\log\pi$, over positions and candidate tokens. They differ only in how the weights $w$ are chosen.** (Our reading, in the spirit of the DeepSeekMath unified view.)

$$\sum_{\tt}\sum_{v} \; w_\tt(v)\;\nabla_\theta \log \pi_\theta(v \mid \y_{<\tt})$$

- $\y_{<\tt}$ is the **prefix**: the prompt plus the completion's tokens before position $\tt$ (RLLM-7)
- $v$ is a **candidate token**: any token the model could write at position $\tt$
  - the sum runs over every candidate because a method may put weight on tokens it did not sample, such as a teacher's distribution
- $\nabla_\theta\log\pi_\theta$ is the Intro's **score function**
  - the **weight** $w$ is its multiplier: what an update puts on a token
  - in policy gradient the weight is the **advantage** $\hat{\advseq}_\tt$
- The network's numbers are the **parameters** $\theta$, never "weights" here

## Gradient form: what the weights look like

> **The weight $w_\tt(v)$ on each candidate token at one position, for three methods.**

<img src="images/weight_patterns.png" alt="the weight on each candidate token at one position, for SFT, PPO or GRPO, and distillation" width=55%>

- **SFT**: weight 1 on the dataset's one token, 0 elsewhere
- **PPO, GRPO**: signed weights on the **sampled** tokens
- **Classical distillation**: training on a **teacher's** saved outputs (a **teacher** is a model we use outputs or probabilities from)
  - the model being trained is the **student**
  - usually stronger; RLLM-11 and RLLM-12 ask when that holds
  - with saved logits, every candidate gets a weight
- A **common form**, not a common objective: each method is the gradient of its own loss


## Push: what a weight does

> **A positive weight pushes a token's probability up, a negative one pushes it down. The rest of the session asks where the weights come from, and how fine they are.**

<img src="images/push_defined.png" alt="weights +1, 0, -1 on three tokens, and the probabilities before and after the update" width=60%>

- **Push**: the direction an update moves the model's probabilities, set by the sign of the weight
- RLLM-7 settled where samples come from. The weights are the other half of an update
- Next: a case where the weights are all equal, and what that does


## No-weights case: equal weights on the model's own samples

> **Why must the weights come from outside? Try the simplest case: the model trains on its own samples with equal weights.**
>
> **This is a thought experiment with an exact answer (a lemma in expectation), not a method. It is the reason axis 2 exists.**

**Predict before you advance the slide.** The completions are drawn at
temperature 1 (the model's own probabilities, unfiltered) from the model itself.
Think of the average over many batches (the next slide says what "unchanged" means). After this one update, the model is:

- **better**: it trained on its own (reasonable) completions
- **worse**: it trained on its own mistakes as well as its successes
- **unchanged, on average**


## No-weights case: no push on average, until a reward breaks it

> **With equal weights, the model's own completions push no logit up or down on average (left). Weights from a reward give a push with a direction (right).**

<img src="images/zero_mean_gradient.png" alt="same toy policy: with equal weights the average push is zero; with weights equal to a noisy reward minus a baseline it points toward the better action" width=46%>

- **Why zero:** probabilities sum to 1. Raising some tokens lowers others, so the pushes cancel
  - **exact in expectation** for a summed sequence loss, at temperature 1 (sampling from the model's own probabilities), with no truncation
  - one finite step is only first order

## No-weights case: what breaks the zero, in practice

- A **finite batch** is noisy (the dots)
- **Adam** (the usual optimizer, which rescales each step) and drift can sharpen the model over steps (entropy falls)
- **Temperature below 1** (sampling more sharply) or **top-p** (sampling only from the most likely tokens whose probabilities add to $p$) sharpens it systematically
- A token-**mean** loss ($1/|\y|$) breaks the zero. It is GRPO's **Averaging** change in RLLM-6: how the loss is averaged over tokens
- So "unchanged" means: **no systematic direction**, not no movement at all

**Reading the figure**

- **How to read it:** a toy policy picks among 4 actions
  - a **logit** is an action's pre-softmax number
  - each *dot* is the push from one batch of 20 completions
- Logit 2 is the best action. Toy numbers: it is pushed up in **83%** of batches with a reward
  - against **45%** with equal weights: no systematic direction (a batch is noisy), not "below a coin flip"
- A reward-based weight is a **noisy reward minus a baseline** (the average reward, as in the Intro)
- So something **outside** the model must supply the weights: a dataset, a teacher, a checker, a reward model (RLLM-2). Who? That is axis 2

> **Keep this: equal weights on your own completions give no push on average. It returns in RLLM-9 as the reason for a filter.**
<!-- seed-plant: zero-push-equal-weights -->


## Weight source: a reference to imitate, or a reward to maximize

> **Axis 2 asks who supplies the weights: a reference (a dataset's token, or a teacher's next-token distribution or the token-level signal read from it, RLLM-11) or a reward (a score or a preference pair).**
>
> **A plain dataset reference also lowers the other tokens, by normalization. What it cannot do is mark a specific *sampled* sequence as bad.**

*Warning: this **reference** is not RLLM-2 and RLLM-4's **reference model** (the frozen KL anchor). Here it means what the update imitates: a dataset's answer, or a teacher's distribution.*

- **SFT**: the reference is a dataset; all the weight sits on its one token
- **Classical distillation** (training on a **teacher's** saved outputs): with saved logits the reference is the teacher's distribution per token. The **student** is the model trained
- **PPO, GRPO**: a reward (usually of the whole completion) becomes **signed** weights on the sampled tokens, after subtracting a baseline
- What you **hold** picks the side: demonstrations give a reference, a checker a reward. RL needs the model to succeed sometimes (RLLM-1)
- Two cases wait for later sessions
  - a **hybrid** (RLLM-9): a reward only **picks the data**, and a reference (the SFT loss) trains on it
  - a **preference pair** (RLLM-10): a better and a worse completion of one prompt


## Weight granularity: how fine are the weights?

> **Axis 3 asks how many numbers the weights carry: one per completion, one per sampled token, or a distribution per token.**

<img src="images/granularity_strip.png" alt="one episode reward per completion, a number per token, a distribution per token" width=62%>

- **One per completion** (the episode reward): one number for a whole completion of $L$ tokens, one episode as in Part 1
  - it makes **credit assignment** hard: which of the $L$ tokens earned it?
  - **GRPO** (RLLM-6) is this coarse: every token gets the same group-relative advantage
- **One per sampled token**: a number for each token the model wrote
- **A distribution per token**: a weight on every candidate token, not only the sampled one
  - **SFT** is the one-hot case
- **Granularity is not the source** (axis 2): the same granularity can come from different sources
  - per token: a **critic** with GAE in **PPO** (RLLM-4, a model to train), a process reward model (RLLM-5), or a teacher's signed signal
  - a distribution per token: a teacher's distribution
- Next: the map, with its cells filled


## Placements: a first map, four methods and the named slots

> **This is the session's empty map, filled in. Axes 1 and 2 form the grid. SFT, classical distillation, PPO and GRPO are placed; later sessions fill the dashed slots by the same axes.**

<img src="images/first_map.png" alt="rows: others' samples and own samples; columns: a reference and a reward supply the weights; four methods placed, and three dashed slots labelled RLLM-9, RLLM-10 and RLLM-11" width=56%>


**Reading the map**


- **Rows**: whose samples (RLLM-7). **Columns**: who supplies the weights, a reference or a reward
- **The third axis is not a grid axis: it is written inside each box**
  - how many numbers the weights carry: one per completion, one per sampled token, or a distribution per token
- **Dashed slots**: RLLM-9, RLLM-10 and RLLM-11, each filled by the same axes
  - RLLM-9: own samples, across both columns (a **hybrid**: a reward picks the data, a reference trains on it)
  - RLLM-10: others' samples, a reward. RLLM-11: own samples, a reference
- A **hybrid on others' samples** also exists: a checker keeps some of another model's completions, then SFT
  - it lies in the others' row. The hybrid box here is the own-samples one, RLLM-9's
- **Own samples with a reference** is not SFT on your own tokens: a teacher supplies the reference at the student's own prefix (RLLM-11)
- The placements are **our reading** of how each method is built, not a result


## Think about

1. You hold demonstrations and a checker for one task. Which axis-2 column is each in?
   - what must be true of the model for the checker's weights to be usable (RLLM-1)?
2. Replace the reward function with a reward *model*. Does the answer on any of the three axes change? Argue each.
3. In the RLLM-11 slot (own samples, a reference supplied), *where* would the reference come from? What would you need in hand?
4. All the completions in a batch earn the same reward. After subtracting the average, what are the weights? What happens to the update?


## What we established, and what it sets up

- **Established**: one **gradient form**, differing in the **weights**. Own completions with **no reward** push nowhere on average
- **Limit of this session**: a placement says **who supplies the weights**, not whether it works. The map still has **named slots**
- **Sets up**: RLLM-9, Sample, filter, imitate: self-training
  - methods that **look like SFT** but pick their data by a reward: a filter is a 0/1 weight chosen by a reward


## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Shao et al. (2024), *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models*](https://arxiv.org/abs/2402.03300): abstract, sections 4.1 and 5.2.1
  - [Nathan Lambert, *Reinforcement Learning (Motivation & Math)*, lecture slides, rlhfbook.com course](https://rlhfbook.com/teach/course/lec3-chap6-p1/): the policy-gradient derivation, REINFORCE, PPO and GRPO in one slide deck
- **Additional** (4 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved


## Aside, justification: why the average push is zero

*For your own review; not worked through in session.*

- The average push is $\mathbb{E}_{\y\sim\pi_\theta}\big[\nabla_\theta\log\pi_\theta(\y)\big]=\sum_\y \pi_\theta(\y)\,\nabla_\theta\log\pi_\theta(\y)$
- **Log-derivative trick** (Intro B1, RLLM-4): $\nabla\log\pi=\nabla\pi/\pi$, so the $\pi$'s cancel: $\sum_\y\nabla_\theta\pi_\theta(\y)=\nabla_\theta\sum_\y\pi_\theta(\y)=\nabla_\theta 1=0$
  - the identity behind the policy gradient, read the other way
- **Toy check** (softmax over 4 actions, parameters = logits $z$): $\nabla_z\log\pi(a)=e_a-\pi$, with $e_a$ one-hot on the sampled action
  - averaging: $\sum_a\pi(a)(e_a-\pi)=\pi-\pi\cdot 1=0$
- **With non-constant weights** $w(a)$, $\sum_a\pi(a)\,w(a)\,(e_a-\pi)\neq 0$ in general: that is the **policy gradient**
- A **constant** weight factors out and gives zero again: subtracting a baseline adds no bias
- All of this holds per token as well as per completion
